In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tokenizers import Tokenizer
from functools import reduce
import uuid
from datasets import load_dataset

In [ ]:
ds = load_dataset("flydexo/tinyfontaine")

In [ ]:
train_data = '\n'.join(ds['train']['text'])
valid_data = '\n'.join(ds['train']['text'])

In [ ]:
tokenizer = Tokenizer.from_file("tokenizer.json")

In [ ]:
device = torch.device('mps')

In [ ]:
seq_len = 72          # Must be large to capture rhymes and meter
vocab_size = tokenizer.get_vocab_size()     # BPE target size
n_hidden = 256        # Brain power for 17th-century French
n_layers = 3          # Network depth
p = 0.3               # Dropout
batch_size = 64       # Increased to 64 for SGD stability!
epochs = 150          # SGD requires significantly more time than 1-Cycle
alpha = 0.1           # AR 
beta = 0.05           # TAR
momentum = 0.9        # Standard momentum
lr = 0.5              # Starting Learning Rate
wd = 1e-5             # Weight Decay (must remain tiny for pure SGD)

In [ ]:
class LM(torch.nn.Module):
    def __init__(self, vocab_size, n_hidden, n_layers, p_dropout):
        super().__init__()
        k = 1.0 / (2 * n_hidden) ** 0.5
        self.embedding = torch.nn.Parameter(torch.empty(vocab_size, n_hidden).uniform_(-0.1, 0.1))
        self.forgettings = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.input_gates = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.cell_gates = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.outputs = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden, n_hidden).uniform_(-k, k))

        self.b_f = torch.nn.Parameter(torch.ones(n_layers, n_hidden))
        self.b_i = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))
        self.b_c = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))
        self.b_o = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))

        
        self.cell_states = None
        self.hidden_states = None
        self.n_layers = n_layers

        self.p = 1-p_dropout
        self.bern = torch.distributions.bernoulli.Bernoulli(self.p)
        self.activations_before_dropout = []
        self.activations_after_dropout = []

    def forward(self, x):
        output = []
        self.activations_before_dropout = []
        self.activations_after_dropout = []
        if self.hidden_states == None or self.cell_states == None:
            self.cell_states = [torch.zeros(x.shape[0],n_hidden,device=device) for _ in range(n_layers)]
            self.hidden_states = [torch.zeros(x.shape[0],n_hidden,device=device) for _ in range(n_layers)]

        masks = None
        if self.training:
            masks = [self.bern.sample((x.shape[0], n_hidden)).to(device) / self.p for _ in range(self.n_layers + 1)] 
        
        for i in range(x.shape[1]):
            for j in range(self.n_layers):
                if j == 0:
                    x_t = self.embedding[x[:,i]]
                else:
                    x_t = self.hidden_states[j-1]

                if self.training:
                    #mask = self.bern.sample(x_t.shape).to(device)
                    x_t = x_t*masks[j]
                    
                input = torch.cat([self.hidden_states[j], x_t], dim=1)
                #input has a shape of batch_size*(2*n_hidden)
                forget_activation = F.sigmoid(input@self.forgettings[j]+self.b_f[j])
                input_activation = F.sigmoid(input@self.input_gates[j]+self.b_i[j])
                cell_activation = F.tanh(input@self.cell_gates[j]+self.b_c[j])
                output_activation = F.sigmoid(input@self.outputs[j]+self.b_o[j])
                self.cell_states[j] = self.cell_states[j]*forget_activation+input_activation*cell_activation
                self.hidden_states[j] = F.tanh(self.cell_states[j])*output_activation
                if j == self.n_layers-1:
                    final_out = self.hidden_states[j]
                    
                    if self.training:
                        self.activations_before_dropout.append(final_out)
                        #mask = self.bern.sample(final_out.shape).to(device)
                        final_out = final_out*masks[-1]
                        self.activations_after_dropout.append(final_out)
                    else:
                        self.activations_before_dropout.append(final_out)
                        self.activations_after_dropout.append(final_out)
                    
                    output.append(final_out@self.embedding.t())
        self.hidden_states = [s.detach() for s in self.hidden_states]
        self.cell_states = [s.detach() for s in self.cell_states]
        return torch.stack(output, dim=1)

    def reset(self):
        self.hidden_states = None
        self.cell_states = None

In [ ]:
def loss_fn(model, pred, target, split):
    loss = torch.nn.CrossEntropyLoss()
    loss_y = loss(pred.transpose(1,2), target)
    if split == "train":
        loss_y += alpha * torch.stack(model.activations_after_dropout).pow(2).mean()
        activations = torch.stack(model.activations_before_dropout)
        loss_y += beta * (activations[1:, :, :] - activations[:-1, :, :]).pow(2).mean()
    #loss_y += wd * (torch.stack(model.parameters())**2).sum()
    return loss_y

In [ ]:
class FontaineDataset(torch.utils.data.IterableDataset):
    def __init__(self, data, seq_len, batch_size):
        super().__init__()
        text_tensor = torch.tensor(tokenizer.encode(data).ids, device=device)
        
        n_tokens = len(text_tensor) - 1 
        tokens_per_stream = n_tokens // batch_size
        
        x_data = text_tensor[:batch_size * tokens_per_stream]
        y_data = text_tensor[1 : batch_size * tokens_per_stream + 1]
        
        x_data = x_data.view(batch_size, -1)
        y_data = y_data.view(batch_size, -1)
        
        self.batches = []
        
        for i in range(0, x_data.shape[1] - seq_len + 1, seq_len):
            x_chunk = x_data[:, i:i+seq_len]
            y_chunk = y_data[:, i:i+seq_len]
            
            if x_chunk.shape[1] == seq_len:
                self.batches.append((x_chunk, y_chunk))

    def __iter__(self):
        return iter(self.batches)

In [ ]:
ds       = FontaineDataset(train_data, seq_len, batch_size)
valid_ds = FontaineDataset(valid_data, seq_len, batch_size)

In [ ]:
model = LM(vocab_size, n_hidden, n_layers, p).to(device)

In [ ]:
velocities = [torch.zeros_like(param) for param in model.parameters()]

In [ ]:
for e in range(93,epochs):
    model.train()
    total_loss, n = 0.0, 0

    for x,y in ds:
        if x.shape == torch.Size([batch_size, seq_len]) and y.shape == torch.Size([batch_size, seq_len]):
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            loss = loss_fn(model, pred, y, "train")
            loss.backward()
            total_loss += loss.item() * x.size(0); n += x.size(0)

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
            with torch.no_grad():
                for param, v in zip(model.parameters(), velocities):
                    g = param.grad + wd * param.data
                    v.mul_(momentum).add_(g)
                    param.data.add_(v, alpha=-lr)
                    param.grad.zero_()
    total_loss /= n
    model.reset()
    model.eval()
    with torch.no_grad():
        vloss = torch.zeros(1, device=device)
        l = 0
        for x,y in valid_ds:
            if x.shape == torch.Size([batch_size, seq_len]) and y.shape == torch.Size([batch_size, seq_len]):
                l+=1
                x = x.to(device)
                y = y.to(device)
                pred = model(x)
                vloss += loss_fn(model, pred, y, "test")
        vloss /= l
    model.reset()
    print(f"epoch {e+1}, training loss: {total_loss:.4f}, validation loss: {float(vloss.data):.4f}")

In [ ]:
torch.save(model, f'{str(uuid.uuid4())}.pkl')

In [ ]:
import torch.nn.functional as F

def generate_fable(model, prompt, n_words=50, temperature=0.8):
    model.eval()
    model.reset() # Start with a clean memory state
    
    # Encode the initial prompt
    # Note: If you switched to the BpeTrainer, you will use tokenizer.encode(prompt).ids here instead!
    tokens = tokenizer.encode(prompt).ids
    
    # Pass the prompt through to build up the hidden states
    x = torch.tensor([tokens]).to(device)
    with torch.no_grad():
        out = model(x)
        
    # Grab the raw logits (unnormalized predictions) for the final word
    logits = out[0, -1, :]
    
    generated_tokens = []
    
    for _ in range(n_words):
        # 1. APPLY TEMPERATURE
        # Temperature < 1.0 makes the model more confident and rigid
        # Temperature > 1.0 makes the model chaotic and highly creative
        scaled_logits = logits / temperature
        
        # 2. CONVERT TO PROBABILITIES
        probs = F.softmax(scaled_logits, dim=-1)
        
        # 3. ROLL THE DICE (Multinomial Sampling)
        # Instead of argmax, we sample from the probability distribution
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated_tokens.append(next_token)
        
        # 4. Feed the new token back into the model to update the memory
        x = torch.tensor([[next_token]]).to(device)
        with torch.no_grad():
            out = model(x)
            logits = out[0, -1, :] # Grab the new logits for the next loop
            
    # Decode the tokens back to text
    # Note: If using BPE, use tokenizer.decode(generated_tokens) instead!
    generated_text = tokenizer.decode(generated_tokens)
    
    # Optional: Clean up standard word-level punctuation spacing
    full_text = f"{prompt} {generated_text}"
    full_text = full_text.replace(" . ", ".\n").replace(" , ", ", ")
    
    return full_text

# Let's write a fable!
# Try tinkering with the temperature. 
# 0.5 will be very repetitive, 1.2 will invent words, 0.8 is usually the sweet spot.
print(generate_fable(model, "<|titre|>L'âne et la fourmi<|titre|>", n_words=1000, temperature=0.1))